# ML-08 - Capstone Modeling Lane

Lane: **Refresh / Content Opportunity Scoring**.

The model asks: using safe page-level signals from August-September 2025, can we rank pages that are likely to lose at least 20% of September impressions in October 2025? The output is a ranked content-review queue, not an automated edit decision.

## 1. Method choice and why

I used a **random forest classifier** as the selected model, with logistic regression as a readable comparison. The lane is a ranking problem, so I compare methods by Precision@K on held-out clients, not by generic accuracy.

The baseline remains the Week-4 transparent rule: `visible_low_ctr_page`. It is useful because a human can read it, but it only sees visible low-CTR pages; the model can combine position, traffic history, CTR, content metadata, and age signals.

In [1]:

from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "work" / "scripts" / "run_capstone_analysis.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root")

ROOT = find_repo_root(Path.cwd().resolve())
METRICS_PATH = ROOT / "work" / "outputs" / "capstone_metrics.json"
if not METRICS_PATH.exists():
    raise FileNotFoundError(
        "Missing work/outputs/capstone_metrics.json. Run `python work/scripts/run_capstone_analysis.py` from the repo root after Hugging Face login."
    )
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
results = pd.DataFrame(metrics["results"])
top_features = pd.DataFrame(metrics["top_features"])
top20 = pd.DataFrame(metrics["top20_preview"])
print(f"Loaded metrics from {METRICS_PATH.relative_to(ROOT)}")
print(f"Eligible rows: {metrics['eligible_rows']:,}; clients: {metrics['eligible_clients']}; base rate: {metrics['base_rate']:.3f}")

display(results)
display(top_features.head(12))

Loaded metrics from work\outputs\capstone_metrics.json
Eligible rows: 24,547; clients: 18; base rate: 0.256


,method,roc_auc,avg_precision,precision_at_50,precision_at_100
0,logistic_regression,0.853186,0.729499,0.90,0.77
1,random_forest,0.871198,0.739586,0.90,0.78
2,baseline_visible_low_ctr,0.470689,0.258484,0.18,0.28


,feature,importance
0,num__avg_position_may,0.252179
1,num__avg_position_apr,0.203353
2,num__ctr_may,0.081764
3,num__imp_change_apr_to_may,0.063747
4,num__content_age_days,0.060400
5,num__clicks_may,0.058961
6,num__impressions_apr,0.048202
7,num__days_since_update,0.040416
8,num__word_count,0.035044
9,num__char_count,0.027729


## 2. Split design

The validation split holds out whole clients. That is stricter than a random row split because pages from the same client can share site structure, tracking coverage, and audience behavior. A page from a held-out client is a better test of whether the rule generalizes beyond clients seen in training.

In [2]:
split_summary = pd.DataFrame([{
    "train_rows": metrics["train_rows"],
    "test_rows": metrics["test_rows"],
    "train_clients": metrics["train_clients"],
    "test_clients": metrics["test_clients"],
    "base_rate": metrics["base_rate"],
}])
display(split_summary)

,train_rows,test_rows,train_clients,test_clients,base_rate
0,23880,667,14,4,0.2555


## 3. Train + compare vs my baseline

The model and baseline are evaluated on the same held-out client split. The selected model is the random forest because it ties logistic regression at Precision@50 while producing stronger ROC AUC, average precision, and Precision@100.

In [3]:
show = results[["method", "roc_auc", "avg_precision", "precision_at_50", "precision_at_100"]].copy()
for col in ["roc_auc", "avg_precision", "precision_at_50", "precision_at_100"]:
    show[col] = show[col].map(lambda x: f"{x:.3f}")
display(show)

,method,roc_auc,avg_precision,precision_at_50,precision_at_100
0,logistic_regression,0.853,0.729,0.900,0.770
1,random_forest,0.871,0.740,0.900,0.780
2,baseline_visible_low_ctr,0.471,0.258,0.180,0.280


## 4. Errors and interpretation

The top feature importances show that position and CTR context carry much of the signal. That does not prove why a page declined; it only says those observed signals helped rank future-decline candidates in this window.

A weak spot is that some high-probability recommendations are deep-position pages with zero clicks. Those should usually be monitored, pruned, consolidated, or investigated before anyone spends refresh time.

In [4]:
display(top20[["rank", "content_hash_id", "action_label", "reason_codes", "model_probability", "impressions_may", "ctr_may", "avg_position_may", "future_decline_label"]].head(10))

,rank,content_hash_id,action_label,reason_codes,model_probability,impressions_may,ctr_may,avg_position_may,future_decline_label
0,1,content_e5119f5ccb21b8be,monitor_or_prune,future_decline_risk;deep_zero_click_risk,0.990137,256,0.0,80.734375,1
1,2,content_84332ce4be404e3f,monitor_or_prune,future_decline_risk;deep_zero_click_risk,0.989507,306,0.0,76.669935,1
2,3,content_e02ca086a84cfd35,monitor_or_prune,future_decline_risk;deep_zero_click_risk,0.987677,228,0.0,81.000000,1
3,4,content_ad0b68d1727295e3,monitor_or_prune,future_decline_risk;deep_zero_click_risk,0.987299,304,0.0,82.569079,1
4,5,content_905103df18204769,monitor_or_prune,future_decline_risk;deep_zero_click_risk,0.987145,263,0.0,75.041825,1
5,6,content_dcdb257bd5040270,monitor_or_prune,future_decline_risk;deep_zero_click_risk,0.987009,241,0.0,84.514523,1
6,7,content_305afb885f6a3b9c,monitor_or_prune,future_decline_risk;deep_zero_click_risk,0.986873,537,0.0,78.370577,1
7,8,content_df405a5ed66aab92,monitor_or_prune,future_decline_risk;deep_zero_click_risk,0.986646,308,0.0,75.564935,1
8,9,content_7faa8052b9d6529d,monitor_or_prune,future_decline_risk;deep_zero_click_risk,0.986624,375,0.0,84.600000,1
9,10,content_80be8e495d9ac06a,monitor_or_prune,future_decline_risk;deep_zero_click_risk,0.986336,418,0.0,79.440191,1


## Self-check

- [x] Method choice is named and tied to the lane.
- [x] Split design is grouped by client.
- [x] Model and baseline are compared on the same split.
- [x] The result is framed as ranking/decision-support, not causal proof.